# 13 Adaptive Query Execution (AQE)

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Understand how AQE re-optimizes a query <b>while it runs</b>, using real shuffle statistics: coalescing small shuffle partitions, switching join strategies, and splitting skewed partitions. Know how to see its effects, and which settings control it.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Static plans are built from estimates, and estimates are often wrong after filters, joins and aggregations. AQE fixes many of those mistakes automatically, which is why default settings work well on Databricks. Interviewers ask "what is AQE and what does it do?", and it's the first answer to "one task is much slower than the others".
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 How AQE works</b><br>
Without AQE, Spark fixes the whole physical plan before running. With AQE (on by default since Spark 3.2, and always on for Databricks serverless):
<ol><li>Spark runs the plan <b>stage by stage</b>, up to each shuffle</li><li>When a shuffle's map stage finishes, it knows the <b>real</b> size of every shuffle partition</li><li>It <b>re-optimizes</b> the rest of the plan with those statistics, then continues</li></ol>
Three main optimizations:<br>| Optimization | Problem it fixes | What AQE does |<br>|---|---|---|<br>| <b>Coalesce shuffle partitions</b> | 200 tiny shuffle partitions → 200 tiny tasks | Merges neighbouring small partitions up to the advisory size (64 MB by default) |<br>| <b>Switch join strategy</b> | Sort-merge join chosen, but one side turns out small | Converts to a broadcast hash join (or shuffled hash join) at runtime |<br>| <b>Skew join</b> | One huge partition makes one task run for ages | Splits the skewed partition into smaller ones and duplicates the matching side |
</div>

```
STATIC PLAN                         ADAPTIVE PLAN
plan everything ──▶ run             run stage 1 ──▶ measure shuffle sizes ──▶ re-plan ──▶ run stage 2 ──▶ ...
(estimates only)                    (real numbers after each shuffle)
```

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A nightly join of orders to customers ran with 200 shuffle partitions and a sort-merge join. One customer, a marketplace account, owned 30% of all orders, so one task ran for 50 minutes while 199 finished in a minute. After upgrading to a runtime with AQE skew handling on, that partition was split into 12 pieces. The stage finished in 6 minutes with no code change.
</div>

## Key settings

| Setting | Default | Meaning | Serverless |
|---|---|---|---|
| `spark.sql.adaptive.enabled` | `true` | Turn AQE on | Always on |
| `spark.sql.adaptive.coalescePartitions.enabled` | `true` | Merge small shuffle partitions | Managed |
| `spark.sql.adaptive.advisoryPartitionSizeInBytes` | `64MB` | Target size when coalescing or splitting | Managed |
| `spark.sql.adaptive.skewJoin.enabled` | `true` | Split skewed join partitions | Managed |
| `spark.sql.adaptive.skewJoin.skewedPartitionFactor` / `...ThresholdInBytes` | `5` / `256MB` | A partition is skewed if it's > factor × median **and** > threshold | Managed |
| `spark.sql.adaptive.autoBroadcastJoinThreshold` | (same as static) | Size below which AQE converts to broadcast | Managed |

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
On serverless you can't change these settings. The cells below try to, and print a note if they're locked. You'll see AQE's results either way, and the query profile shows the final, adapted plan.
</div>

## Setup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Defines helpers: one to set a config safely, and one to count the partitions that hold data.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
No output.
</div>

In [0]:
import time
from pyspark.sql import functions as F

def try_set(key, value):
    try:
        spark.conf.set(key, value)
        return True
    except Exception:
        return False

def partitions_with_data(df):
    return df.select(F.spark_partition_id()).distinct().count()

original_shuffle = spark.conf.get("spark.sql.shuffle.partitions")

## 1. AQE in the plan

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Prints the plan of a simple aggregation.<br><br>
<b>Why it matters</b><br><code>AdaptiveSparkPlan isFinalPlan=false</code> at the top means the plan you see is the <b>initial</b> one. AQE will change it as stages complete. The final plan is shown in the Spark UI's SQL tab (classic) or the query profile (serverless).
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A plan starting with <code>AdaptiveSparkPlan isFinalPlan=false</code>.
</div>

In [0]:
spark.range(1_000_000).groupBy((F.col("id") % 10).alias("k")).count().explain()

## 2. Coalescing shuffle partitions

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Groups 2 million rows into 10 keys with 200 shuffle partitions, with AQE coalescing off and on, and counts the partitions that hold data.<br><br>
<b>Why it matters</b><br>Without AQE, a reduce task runs for each of the 200 partitions, even the empty ones. With AQE, small partitions are merged after the map stage, so only a handful of tasks run.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Coalescing off (if allowed): up to 10 partitions with data out of 200 tasks. Coalescing on: 1 partition with data, because the 10 tiny groups were merged.
</div>

In [0]:
spark.conf.set("spark.sql.shuffle.partitions", "200")
query = spark.range(2_000_000).groupBy((F.col("id") % 10).alias("k")).count()

if try_set("spark.sql.adaptive.coalescePartitions.enabled", "false"):
    print("coalescing OFF -> partitions with data:", partitions_with_data(query))
    try_set("spark.sql.adaptive.coalescePartitions.enabled", "true")
else:
    print("coalescing setting is managed here")
print("coalescing ON  -> partitions with data:", partitions_with_data(query))
spark.conf.set("spark.sql.shuffle.partitions", original_shuffle)

## 3. Skew join handling

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Joins a skewed table (80% of rows share one key) with a sort-merge join, with AQE skew handling off and on, and counts the partitions of the join output. Thresholds are lowered so this small demo counts as "skewed".<br><br>
<b>Why it matters</b><br>With skew handling on, AQE splits the huge partition into several smaller ones (and duplicates the matching rows from the other side), so the work is spread over more tasks. The extra output partitions are the visible sign.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Where the settings can be changed: about 8 partitions with skew handling off, and noticeably more (for example 15) with it on, plus timings. On serverless, a note that the settings are managed. There the query profile shows a skew-split join when the data is large enough.
</div>

In [0]:
skewed = spark.range(0, 2_000_000, 1, numPartitions=8).withColumn(
    "key", F.when(F.col("id") % 10 < 8, F.lit(0)).otherwise(F.col("id") % 1000))
dim = spark.range(1000).withColumnRenamed("id", "key").withColumn("label", F.concat(F.lit("k"), F.col("key")))

spark.conf.set("spark.sql.shuffle.partitions", "8")
demo_settings = {
    "spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes": "1k",
    "spark.sql.adaptive.advisoryPartitionSizeInBytes": "64k",
    "spark.sql.adaptive.coalescePartitions.enabled": "false",
}
if all(try_set(k, v) for k, v in demo_settings.items()):
    for enabled in ["false", "true"]:
        try_set("spark.sql.adaptive.skewJoin.enabled", enabled)
        joined = skewed.join(dim.hint("merge"), "key")
        start = time.time()
        n = partitions_with_data(joined)
        print(f"skew handling {'ON ' if enabled == 'true' else 'OFF'} -> join output partitions: {n:>3}  ({time.time() - start:.2f}s)")
    for key, value in [("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "256MB"),
                       ("spark.sql.adaptive.advisoryPartitionSizeInBytes", "64MB"),
                       ("spark.sql.adaptive.coalescePartitions.enabled", "true"),
                       ("spark.sql.adaptive.skewJoin.enabled", "true")]:
        try_set(key, value)
else:
    print("AQE skew settings are managed on this compute. Run the join and open its query profile:")
    skewed.join(dim.hint("merge"), "key").write.format("noop").mode("overwrite").save()
spark.conf.set("spark.sql.shuffle.partitions", original_shuffle)

## 4. Switching a sort-merge join to a broadcast join

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Joins a large table to a table that is large <b>before</b> filtering but tiny <b>after</b>, and shows that the initial plan is a sort-merge join.<br><br>
<b>Why it matters</b><br>The static estimate can't know how selective the filter is, so it plans a sort-merge join. When the shuffle of the filtered side finishes and turns out to be tiny, AQE converts the join to a broadcast hash join (and uses a local shuffle reader to avoid redundant work). This conversion is only visible in the <b>final</b> plan, in the query profile or Spark UI.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The initial plan shows <code>SortMergeJoin</code>. After running, open the query profile (serverless) or the SQL tab (classic) and find <code>BroadcastHashJoin</code> in the final plan.
</div>

In [0]:
facts = spark.range(0, 3_000_000, 1, numPartitions=8).withColumn("cust", F.col("id") % 5_000_000)
customers = (
    spark.range(5_000_000).withColumnRenamed("id", "cust")
    .withColumn("name", F.concat(F.lit("customer_"), F.col("cust").cast("string")))
    .withColumn("vip", F.col("cust") % 25_000 == 0)
)
vips = customers.filter("vip")                   # 200 rows, but the static estimate is close to the full table

import io, contextlib

adaptive_join = facts.join(vips, "cust")
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    adaptive_join.explain()
initial = buf.getvalue()
print("initial plan join:", "SortMergeJoin" if "SortMergeJoin" in initial else "BroadcastHashJoin" if "BroadcastHashJoin" in initial else "other")
print(initial)
start = time.time()
print("rows:", adaptive_join.count(), f"({time.time() - start:.2f}s)")

## Under the hood

```
1. Planner builds the initial physical plan, wrapped in AdaptiveSparkPlan
2. AQE splits it into "query stages" at every Exchange and submits the leaves
3. Each finished map stage reports MapOutputStatistics (bytes per shuffle partition)
4. AQE re-optimizes the remaining plan:
     - CoalesceShufflePartitions      merge small partitions up to advisory size
     - OptimizeSkewedJoin             split partitions > factor × median and > threshold
     - DynamicJoinSelection / demote  SMJ → BHJ when a side is small enough
     - OptimizeShuffleWithLocalRead   read shuffle output locally after a switch
5. Repeat until the final stage; the Spark UI shows isFinalPlan=true
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Wide</span> AQE works at shuffle boundaries: it needs a completed shuffle to measure.

**Limits:** AQE can't fix skew inside a single stage without a shuffle, skew in aggregations (only joins are split), or a bad data layout. And it can only coalesce or split **after** the first shuffle. File-read partitions are decided by `maxPartitionBytes`.

In [0]:
spark.range(2_000_000).groupBy((F.col("id") % 3).alias("k")).agg(F.sum("id")).explain("formatted")

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: <code>explain()</code> doesn't show the optimization I expected</b><br>
It shows the initial plan (<code>isFinalPlan=false</code>). Check the final plan in the query profile or Spark UI.<br><br>
<b>⛔ Problem: skew isn't handled</b><br>The partition isn't large enough to count as skewed (factor and threshold), or the skew is in an aggregation rather than a join. Salt the key (<code>05_optimization/02_data_skew</code>).<br><br>
<b>⛔ Problem: still many small tasks after a shuffle</b><br>Coalescing is disabled, or the advisory size is too small.<br><br>
<b>⛔ Problem: turning AQE off "to make results reproducible"</b><br>AQE doesn't change results, only the physical execution. Keep it on.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is Adaptive Query Execution?</b><br>
A Spark 3 feature that re-optimizes the physical plan at runtime using real statistics from completed shuffle stages, instead of relying only on estimates made before execution.<br><br>

<b>🎤 2. What are the main AQE optimizations?</b><br>
Coalescing small shuffle partitions into fewer, larger ones; switching a sort-merge join to a broadcast hash join when a side turns out small; and splitting skewed partitions in sort-merge joins.<br><br>

<b>🎤 3. How does AQE detect a skewed partition?</b><br>
A partition is skewed if it's larger than a factor (5 by default) times the median partition size and also larger than a threshold (256 MB by default). AQE then splits it and replicates the matching data from the other side.<br><br>

<b>🎤 4. Why does <code>explain()</code> show <code>isFinalPlan=false</code>?</b><br>
Because the plan can still change at runtime. The final adaptive plan is visible in the Spark UI's SQL tab or the Databricks query profile after the query runs.<br><br>

<b>🎤 5. What can't AQE fix?</b><br>
Problems before the first shuffle (such as file-read partitioning or a single huge file), skew in aggregations, poor data layout, and inefficient code such as Python UDFs.<br><br>

<b>🎤 6. Do you still need to tune <code>spark.sql.shuffle.partitions</code> with AQE?</b><br>
Much less. AQE coalesces small partitions, so a reasonably high value works well. For very large shuffles you may still raise it, so the initial partitions aren't too big before coalescing.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. Which Spark feature can automatically convert a sort-merge join into a broadcast join at runtime, after discovering that one side is small?</b><br>
A. Catalyst constant folding<br>
B. Adaptive Query Execution<br>
C. Delta data skipping<br>
D. Whole-stage code generation<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. A join stage has one task processing 40 GB while the others process 200 MB each. Which AQE feature is designed for this?</b><br>
A. Coalescing shuffle partitions<br>
B. Skew join optimization<br>
C. Dynamic partition pruning<br>
D. Predicate pushdown<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Group <code>spark.range(5_000_000)</code> by <code>id % 25</code> with 400 shuffle partitions, and count the partitions with data. Explain the result</li><li>Build a skewed join of your own (90% of rows on one key) and, if your compute allows it, compare the join output partitions with skew handling off and on</li><li>In two sentences, prepare your spoken answer to "What is AQE?"</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. 400 planned partitions, but AQE coalesces the 25 small groups into very few
spark.conf.set("spark.sql.shuffle.partitions", "400")
print("partitions with data:", partitions_with_data(spark.range(5_000_000).groupBy((F.col("id") % 25).alias("k")).count()))
spark.conf.set("spark.sql.shuffle.partitions", original_shuffle)

# 2. 90% skew
very_skewed = spark.range(0, 2_000_000, 1, numPartitions=8).withColumn(
    "key", F.when(F.col("id") % 10 < 9, F.lit(0)).otherwise(F.col("id") % 500))
spark.conf.set("spark.sql.shuffle.partitions", "8")
if all(try_set(k, v) for k, v in demo_settings.items()):
    for enabled in ["false", "true"]:
        try_set("spark.sql.adaptive.skewJoin.enabled", enabled)
        print(f"skew {enabled}:", partitions_with_data(very_skewed.join(dim.hint("merge"), "key")))
    for key, value in [("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "256MB"),
                       ("spark.sql.adaptive.advisoryPartitionSizeInBytes", "64MB"),
                       ("spark.sql.adaptive.coalescePartitions.enabled", "true"),
                       ("spark.sql.adaptive.skewJoin.enabled", "true")]:
        try_set(key, value)
else:
    print("settings managed on this compute")
spark.conf.set("spark.sql.shuffle.partitions", original_shuffle)

# 3. AQE re-optimizes the query while it runs, using the real sizes of completed shuffle stages.
#    It coalesces small partitions, switches sort-merge joins to broadcast joins, and splits skewed join partitions.

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>AQE re-plans at each shuffle boundary using real statistics</li><li>Coalesces small shuffle partitions, switches SMJ → BHJ, splits skewed join partitions</li><li>On by default (Spark 3.2+), always on for serverless</li><li><code>explain()</code> shows the initial plan. The final plan is in the query profile / Spark UI</li><li>Skew = larger than 5 × median and larger than 256 MB (defaults)</li><li>Can't fix pre-shuffle problems, aggregation skew, or bad layout</li></ul>
</div>

| Feature | Visible as |
|---|---|
| Coalescing | Fewer tasks after a shuffle, `AQEShuffleRead coalesced` |
| Join switch | `BroadcastHashJoin` in the final plan |
| Skew split | More tasks in the join stage, `AQEShuffleRead skewed` |
| AQE active | `AdaptiveSparkPlan` at the top of the plan |

## Git commit

```
git add 02_spark_internals/13_adaptive_query_execution.ipynb
git commit -m "add 02_13 adaptive query execution notebook"
```